In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_7.py ──
"""
Shared infrastructure for Exercise 7 — Transfer Learning.

Contains: CIFAR-10 data loading, feature visualisation helpers,
ExperimentTracker/ModelRegistry setup, training harness.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as T

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex7_transfer_learning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — CIFAR-10 (full 50K, resized for ResNet-18)
# ════════════════════════════════════════════════════════════════════════
# ResNet-18 was designed for 224x224 ImageNet images. CIFAR-10 is 32x32.
# We resize to 96x96: ResNet's strided convolutions shrink spatial dims
# by 32x, so 32x32 would collapse to 1x1 before final pooling. 96x96
# gives a 3x3 final feature map — enough spatial information to learn.

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cifar10"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_SIZE = 96
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
N_CLASSES = 10
BATCH_SIZE = 128
EPOCHS = 8

CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

# Standard transforms: ImageNet normalisation + resize for ResNet
train_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.RandomHorizontalFlip(),
        T.RandomCrop(INPUT_SIZE, padding=8),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)
val_transform = T.Compose(
    [
        T.Resize((INPUT_SIZE, INPUT_SIZE)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ]
)


def load_cifar10() -> tuple[
    torchvision.datasets.CIFAR10,
    torchvision.datasets.CIFAR10,
    DataLoader,
    DataLoader,
]:
    """Load CIFAR-10 with ImageNet-style transforms.

    Returns:
        train_set, val_set, train_loader, val_loader
    """
    train_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=train_transform,
    )
    val_set = torchvision.datasets.CIFAR10(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=val_transform,
    )

    train_loader = DataLoader(
        train_set,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=0)

    print(
        f"CIFAR-10 (full): train={len(train_set)}, val={len(val_set)}, "
        f"classes={N_CLASSES}"
    )
    print(f"  Input size: {INPUT_SIZE}x{INPUT_SIZE} (resized for ResNet-18)")
    print(f"  Classes: {CLASS_NAMES}")

    return train_set, val_set, train_loader, val_loader


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_transfer.db"
    registry_db = "sqlite:///mlfp05_transfer_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_transfer_learning", registry, True


def init_engines() -> tuple[
    ConnectionManager,
    ExperimentTracker,
    str,
    ModelRegistry | None,
    bool,
]:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS — shared by all technique files
# ════════════════════════════════════════════════════════════════════════


async def _train_model_async(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Train a model and log everything to ExperimentTracker.

    Returns:
        train_losses, val_accs, train_accs (per-epoch)
    """
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\n-- {name} --  trainable params: {n_trainable:,} / {n_total:,}")

    opt = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    train_losses: list[float] = []
    val_accs: list[float] = []
    train_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "trainable_params": str(n_trainable),
                "total_params": str(n_total),
                "epochs": str(epochs),
                "lr": str(lr),
                "batch_size": str(tr_loader.batch_size),
                "dataset_size": str(len(tr_loader.dataset)),
            }
        )

        for epoch in range(epochs):
            # -- Training --
            model.train()
            batch_losses = []
            correct = total = 0
            for xb, yb in tr_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                logits = model(xb)
                loss = F.cross_entropy(logits, yb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
                correct += int((logits.argmax(dim=-1) == yb).sum().item())
                total += int(yb.size(0))
            train_losses.append(float(np.mean(batch_losses)))
            train_accs.append(correct / total)
            scheduler.step()

            # -- Validation --
            model.eval()
            correct = total = 0
            with torch.no_grad():
                for xb, yb in vl_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    preds = model(xb).argmax(dim=-1)
                    correct += int((preds == yb).sum().item())
                    total += int(yb.size(0))
            val_accs.append(correct / total)

            await run.log_metrics(
                {
                    "train_loss": train_losses[-1],
                    "train_acc": train_accs[-1],
                    "val_acc": val_accs[-1],
                },
                step=epoch + 1,
            )

            print(
                f"  epoch {epoch + 1}/{epochs}  "
                f"loss={train_losses[-1]:.4f}  "
                f"train_acc={train_accs[-1]:.3f}  "
                f"val_acc={val_accs[-1]:.3f}"
            )

        await run.log_metrics(
            {
                "final_val_acc": val_accs[-1],
                "best_val_acc": max(val_accs),
                "final_train_loss": train_losses[-1],
            }
        )

    return train_losses, val_accs, train_accs


def train_model(
    model: nn.Module,
    name: str,
    tracker: ExperimentTracker,
    exp_name: str,
    tr_loader: DataLoader,
    vl_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
) -> tuple[list[float], list[float], list[float]]:
    """Sync wrapper -- one asyncio.run per training call."""
    return asyncio.run(
        _train_model_async(
            model,
            name,
            tracker,
            exp_name,
            tr_loader,
            vl_loader,
            epochs,
            lr,
        )
    )


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Register a trained model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="val_acc", value=val_acc),
            MetricSpec(name="final_loss", value=final_loss),
        ],
    )
    print(f"  Registered {name}: version={version.version}, acc={val_acc:.3f}")
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    val_acc: float,
    final_loss: float,
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, val_acc, final_loss))


# ════════════════════════════════════════════════════════════════════════
# FEATURE EXTRACTION & VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════


def extract_features(
    model: nn.Module,
    loader: DataLoader,
    max_samples: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    """Extract features from the penultimate layer (before fc head).

    Works for both ResNet (avgpool hook) and Sequential models.
    """
    model.eval()
    hook_features: list[torch.Tensor] = []
    labels: list[np.ndarray] = []

    def hook_fn(module, inp, out):
        del module, inp  # PyTorch hook protocol args; not used here
        hook_features.append(out.flatten(1).detach().cpu())

    # ResNet: hook into avgpool; Sequential: second-to-last layer
    if hasattr(model, "avgpool"):
        avgpool = model.avgpool
        assert isinstance(
            avgpool, nn.Module
        ), f"expected nn.Module for avgpool, got {type(avgpool).__name__}"
        handle = avgpool.register_forward_hook(hook_fn)
    else:
        assert isinstance(
            model, nn.Sequential
        ), f"hook fallback requires nn.Sequential, got {type(model).__name__}"
        handle = model[-3].register_forward_hook(hook_fn)

    with torch.no_grad():
        collected = 0
        for xb, yb in loader:
            if collected >= max_samples:
                break
            xb = xb.to(device)
            model(xb)
            labels.append(yb.numpy())
            collected += len(yb)

    handle.remove()
    features_np = torch.cat(hook_features, dim=0).numpy()[:max_samples]
    labels_np = np.concatenate(labels)[:max_samples]
    return features_np, labels_np


def compute_tsne(features: np.ndarray, perplexity: int = 30) -> np.ndarray:
    """Run t-SNE dimensionality reduction to 2D."""
    # sklearn 1.5+ renamed n_iter → max_iter in TSNE.__init__
    tsne = TSNE(n_components=2, perplexity=perplexity, max_iter=500, random_state=42)
    return tsne.fit_transform(features)


def cluster_quality(coords: np.ndarray, labels: np.ndarray) -> float:
    """Cluster quality: ratio of intra-class to inter-class distance (lower = better)."""
    intra = []
    centroids = []
    for c in range(N_CLASSES):
        mask = labels == c
        if mask.sum() < 2:
            continue
        pts = coords[mask]
        centroid = pts.mean(axis=0)
        centroids.append(centroid)
        intra.append(np.mean(np.linalg.norm(pts - centroid, axis=1)))
    centroids_arr = np.array(centroids)
    inter = np.mean(
        [
            np.linalg.norm(centroids_arr[i] - centroids_arr[j])
            for i in range(len(centroids_arr))
            for j in range(i + 1, len(centroids_arr))
        ]
    )
    avg_intra = np.mean(intra)
    return float(avg_intra / inter) if inter > 0 else float("inf")


def plot_tsne(
    coords: np.ndarray,
    labels: np.ndarray,
    title: str,
    output_path: str | Path,
) -> None:
    """Create and save a t-SNE scatter plot coloured by class."""
    fig = go.Figure()
    for c in range(N_CLASSES):
        mask = labels == c
        fig.add_trace(
            go.Scatter(
                x=coords[mask, 0],
                y=coords[mask, 1],
                mode="markers",
                name=CLASS_NAMES[c],
                marker=dict(size=4, opacity=0.6),
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="t-SNE 1",
        yaxis_title="t-SNE 2",
        template="plotly_white",
    )
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def create_visualizer() -> ModelVisualizer:
    """Return a configured ModelVisualizer instance."""
    return ModelVisualizer()


def save_training_plots(
    viz: ModelVisualizer,
    metrics: dict[str, list[float]],
    output_path: str | Path,
) -> None:
    """Save a training history plot to HTML."""
    fig = viz.training_history(metrics=metrics, x_label="Epoch", y_label="Value")
    fig.write_html(str(output_path))
    print(f"  Saved: {output_path}")


def count_params(model: nn.Module, trainable_only: bool = False) -> int:
    """Count parameters in a model."""
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 7, Part 2: Transfer Learning with ResNet-18
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this section, you will be able to:
#   - Load a pre-trained ResNet-18 (ImageNet weights) and freeze its
#     convolutional backbone
#   - Replace the final classifier head for CIFAR-10 (10 classes)
#   - Compare transfer learning vs from-scratch on the same dataset
#   - Visualise layer activations (pretrained vs random init) and
#     use Grad-CAM to see what the model "looks at"
#   - Apply transfer learning to a real medical imaging scenario
#
# PREREQUISITES: Part 1 (from-scratch baseline), M5/ex_2 (CNNs).
# ESTIMATED TIME: ~30 min
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision



## THEORY — Feature Reuse: The Doctor Who Specialises


In [ ]:
# Think of a doctor who studied general medicine for 7 years, then
# specialises in dermatology for 1 year. The general training (anatomy,
# physiology, diagnosis methodology) transfers directly — the doctor
# doesn't re-learn how to read a biopsy from scratch.
#
# Transfer learning works the same way. A ResNet-18 trained on ImageNet
# (1.28M images, 1000 classes) has already learned:
#   - Layer 1-2: Edge detectors, colour gradients (universally useful)
#   - Layer 3-4: Texture patterns, simple shapes (mostly transferable)
#   - Layer 5+:  Object parts, task-specific combinations (less transferable)
#
# We FREEZE these learned features and only train a new classification
# head for our specific task (CIFAR-10's 10 classes). This means:
#   - ~5,000 trainable parameters instead of ~11 million
#   - Training completes in minutes instead of hours
#   - Works well even with small datasets (because features are pre-learned)
#
# What transfers well:
#   - Low-level features (edges, textures) — almost always useful
#   - Mid-level features (patterns, shapes) — usually useful
#
# What doesn't transfer:
#   - Task-specific combinations (ImageNet "cat ears" ≠ CIFAR "cat face")
#   - Domain-specific patterns (medical images ≠ natural photos)



In [ ]:
print("\n" + "=" * 70)
print("  PART 2: Transfer Learning with Frozen ResNet-18")
print("=" * 70)



## TASK 1 — Load data and engines


In [ ]:
train_set, val_set, train_loader, val_loader = load_cifar10()
conn, tracker, exp_name, registry, has_registry = init_engines()

print(f"\n  Using device: {device}")



## TASK 2 — Build the transfer-learning ResNet-18


Build a ResNet-18 with frozen backbone and fresh classifier head.


In [ ]:
# torchvision.models.resnet18(weights=IMAGENET1K_V1) loads the ImageNet
# checkpoint (~44 MB). We freeze all convolutional layers and replace
# the final fc with a fresh 10-class head.


def build_transfer_resnet(
    n_classes: int = N_CLASSES,
    freeze_backbone: bool = True,
) -> nn.Module:
    # TODO: Load pre-trained ResNet-18 and adapt it for CIFAR-10
    # Steps:
    #   1. Load weights: torchvision.models.ResNet18_Weights.IMAGENET1K_V1
    #   2. Create model: torchvision.models.resnet18(weights=weights)
    #   3. Freeze backbone — for param in model.parameters(): param.requires_grad = False
    #   4. Replace model.fc: get in_features = model.fc.in_features,
    #      then model.fc = nn.Linear(in_features, n_classes)
    # Hint: Wrap in try/except for offline fallback (weights=None)
    pass  # Replace with your implementation


transfer_model = build_transfer_resnet()
n_trainable = count_params(transfer_model, trainable_only=True)
n_total = count_params(transfer_model)
print(
    f"  Trainable: {n_trainable:,} / {n_total:,} ({100 * n_trainable / n_total:.2f}%)"
)



### Checkpoint 1


In [ ]:
assert n_trainable < n_total, "Backbone should be frozen (fewer trainable than total)"
assert n_trainable < 15000, f"Only fc head should be trainable, got {n_trainable:,}"
# INTERPRETATION: We're training only ~5K parameters (the new fc head)
# while keeping ~11M parameters frozen. The frozen backbone provides
# powerful feature extraction learned from 1.28M ImageNet images.
print("--- Checkpoint 1 passed --- transfer model built\n")



## TASK 3 — Build the from-scratch CNN for comparison


Baseline: same architecture as Part 1.


In [ ]:
def build_scratch_cnn(n_classes: int = N_CLASSES) -> nn.Module:
    # TODO: Build the same 3-conv-layer CNN from Part 1
    # Hint: nn.Sequential with Conv2d->BN->ReLU->Pool blocks, then
    #       AdaptiveAvgPool2d(1)->Flatten->Dropout(0.3)->Linear(128, n_classes)
    pass  # Replace with your implementation



## TASK 4 — Train both models and compare


In [ ]:
print("\n" + "=" * 70)
print("  TRAINING: Transfer Learning (frozen ResNet-18 + new head)")
print("=" * 70)
transfer_losses, transfer_accs, transfer_train_accs = train_model(
    transfer_model,
    "resnet18_transfer",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)

print("\n" + "=" * 70)
print("  TRAINING: From-Scratch CNN baseline")
print("=" * 70)
scratch_model = build_scratch_cnn()
scratch_losses, scratch_accs, scratch_train_accs = train_model(
    scratch_model,
    "cnn_from_scratch",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)

best_transfer = max(transfer_accs)
best_scratch = max(scratch_accs)



### Checkpoint 2


In [ ]:
assert best_transfer > 0.50, (
    f"Transfer val accuracy {best_transfer:.3f} below 0.50 -- check ImageNet "
    "normalisation, input resize, or epoch count"
)
assert len(transfer_losses) == EPOCHS, "Should have per-epoch losses"
assert len(scratch_losses) == EPOCHS, "Should have per-epoch losses"
# INTERPRETATION: Transfer learning leverages features already learned
# on ImageNet's 1.28M images. Even though we only train a linear head,
# the frozen ResNet-18 backbone provides powerful feature extraction
# that the from-scratch CNN cannot match in the same number of epochs.
print(f"\n  Transfer best val_acc: {best_transfer:.3f}")
print(f"  Scratch  best val_acc: {best_scratch:.3f}")
print(f"  Advantage: {best_transfer - best_scratch:+.3f}")
print("\n--- Checkpoint 2 passed --- both models trained and compared\n")



## TASK 5 — Register models in ModelRegistry


In [ ]:
if has_registry:
    # TODO: Register both models in the ModelRegistry and promote the better one
    # Steps:
    #   1. register_model(registry, "cifar10_resnet18_transfer", transfer_model, ...)
    #   2. register_model(registry, "cifar10_cnn_scratch", scratch_model, ...)
    #   3. Promote the model with higher accuracy to "production" stage
    # Hint: Use register_model() from helpers
    # Hint: For promotion, use registry.promote_model(name=..., version=...,
    #       target_stage="production", reason=...)
    pass  # TODO: Register and promote models
else:
    print("  Note: ModelRegistry not available. Skipping registration.")



### Checkpoint 3


In [ ]:
# INTERPRETATION: The ModelRegistry gives every model a version, metrics,
# and an audit trail. Promoting a model to production records the exact
# comparison that justified the decision.
print("\n--- Checkpoint 3 passed --- models registered\n")



## TASK 6 — Visualise: Layer activations and Grad-CAM


In [ ]:
# Two powerful visualisations reveal HOW transfer learning works:
#
# 1. LAYER ACTIVATION COMPARISON — We compare activations from the
#    pre-trained ResNet (structured, edge-aware) vs a randomly
#    initialised ResNet (noisy, meaningless).
#
# 2. GRAD-CAM — Gradient-weighted Class Activation Mapping shows which
#    regions of the input image the model focuses on for its prediction.
#    This is the "visual explanation" of what the model learned.

print("-- Layer activation comparison --")

# TODO: Compare layer activations between pretrained and random ResNet
# Steps:
#   1. Build a random ResNet: torchvision.models.resnet18(weights=None)
#   2. Get a sample image from val_loader
#   3. Hook into conv1 of both models using register_forward_hook
#   4. Run forward pass on both models
#   5. Visualise first 8 activation maps from each in a Plotly figure
# Hint: def hook_fn(m, inp, out): acts_list.append(out.detach().cpu())
# Hint: handle = model.conv1.register_forward_hook(hook_fn)
# Hint: Remember to call handle.remove() after the forward pass

random_resnet = torchvision.models.resnet18(weights=None)
random_resnet.eval()
random_resnet.to(device)

sample_batch_x, sample_batch_y = next(iter(val_loader))
sample_img = sample_batch_x[0:1].to(device)

# TODO: Set up hooks, run forward passes, collect activations
# TODO: Create fig_acts with Heatmap traces for pretrained vs random
# TODO: Save to OUTPUT_DIR / "02_activation_comparison.html"
pass  # Replace with your activation comparison code

# -- Grad-CAM visualisation --
print("-- Computing Grad-CAM heatmaps --")

# TODO: Compute Grad-CAM for the transfer model
# Steps:
#   1. Enable gradients on sample image: sample_img_grad = sample_img.clone().requires_grad_(True)
#   2. Hook into transfer_model.layer4 (forward + backward hooks)
#   3. Forward pass -> get predicted class
#   4. Backward pass for predicted class: logits[0, pred_class].backward()
#   5. Compute Grad-CAM: weights = grads.mean(dim=[2,3], keepdim=True)
#      cam = (weights * acts).sum(dim=1, keepdim=True)
#      cam = F.relu(cam)  # Only positive contributions
#   6. Normalise to [0,1] and visualise with go.Heatmap
# Hint: fwd_handle = model.layer4.register_forward_hook(fwd_hook)
# Hint: bwd_handle = model.layer4.register_full_backward_hook(bwd_hook)
# Hint: Don't forget to remove hooks after use
pass  # Replace with your Grad-CAM implementation

# -- t-SNE comparison --
print("\n-- t-SNE: Transfer vs Scratch feature spaces --")
transfer_feats, transfer_labels = extract_features(transfer_model, val_loader)
scratch_feats, scratch_labels = extract_features(scratch_model, val_loader)

coords_transfer = compute_tsne(transfer_feats)
coords_scratch = compute_tsne(scratch_feats)

cq_transfer = cluster_quality(coords_transfer, transfer_labels)
cq_scratch = cluster_quality(coords_scratch, scratch_labels)

print(f"  Cluster quality (lower = better):")
print(f"    Transfer: {cq_transfer:.4f}")
print(f"    Scratch:  {cq_scratch:.4f}")

plot_tsne(
    coords_transfer,
    transfer_labels,
    "t-SNE: Transfer Learning Features (ResNet-18)",
    OUTPUT_DIR / "02_transfer_tsne.html",
)

# Training curves comparison
viz = create_visualizer()
save_training_plots(
    viz,
    {
        "transfer loss": transfer_losses,
        "transfer val_acc": transfer_accs,
        "scratch loss": scratch_losses,
        "scratch val_acc": scratch_accs,
    },
    OUTPUT_DIR / "02_training_comparison.html",
)



### Checkpoint 4


In [ ]:
assert transfer_feats.shape[0] > 0, "Should have extracted transfer features"
# INTERPRETATION: The Grad-CAM heatmap shows the model focuses on the
# object in the image, not the background. Pre-trained features give
# structured attention patterns; scratch-trained features attend more
# diffusely. The t-SNE shows tighter class clusters for transfer
# learning because ImageNet pre-training teaches general visual
# category separation.
print("\n--- Checkpoint 4 passed --- visualisations complete\n")



## TASK 7 — Apply: Medical Image Classification at National Skin Centre


In [ ]:
# SCENARIO: The National Skin Centre (NSC) Singapore wants to build an
# AI system to classify skin conditions from dermatology images. They
# have only 2,000 labelled dermatology images across 10 condition types.
#
# Training a CNN from scratch on 2,000 images would give poor results.
# Transfer learning from ImageNet provides a strong foundation: the
# low-level features (edges, textures, colour patterns) transfer well
# to medical images, even though ImageNet doesn't contain dermatology
# photos.

print("\n" + "=" * 70)
print("  APPLY: Medical Image Classification — National Skin Centre SG")
print("=" * 70)

# TODO: Simulate the medical scenario with only 2,000 labelled images
# Steps:
#   1. Create a random subset of 2,000 indices from train_set
#   2. Build a DataLoader for the medical subset
#   3. Train a transfer model (build_transfer_resnet()) on 2K images
#   4. Train a scratch model (build_scratch_cnn()) on 2K images
#   5. Compare accuracy and calculate cost savings
# Hint: rng = np.random.default_rng(42)
# Hint: n_medical = 2000
# Hint: Same pattern as the startup scenario in Part 1

from torch.utils.data import Subset, DataLoader as DL

rng = np.random.default_rng(42)
n_medical = 2000
# TODO: Create indices, subset, loader
# TODO: Train medical_transfer model
# TODO: Train medical_scratch model
# TODO: Store results in best_medical_transfer and best_medical_scratch
best_medical_transfer = 0.0  # TODO: Replace after training
best_medical_scratch = 0.0  # TODO: Replace after training

print(f"\n  === National Skin Centre Scenario (2,000 images) ===")
print(f"  {'Method':<25} {'Val Accuracy':>15} {'Trainable Params':>18}")
print("  " + "-" * 60)
print(
    f"  {'Transfer (ResNet-18)':<25} " f"{best_medical_transfer:>15.1%} " f"{'~5K':>18}"
)
print(f"  {'From scratch':<25} " f"{best_medical_scratch:>15.1%} " f"{'~200K':>18}")
print(f"  {'Advantage':<25} {best_medical_transfer - best_medical_scratch:>+15.1%}")
print()
print(f"  COST-BENEFIT ANALYSIS:")
print(f"  Labelling cost per image (dermatologist review): ~S$5.00")
print(f"  Current labelled images: 2,000")
print(f"  To match transfer accuracy from scratch: ~20,000+ images needed")
print(f"  Labelling cost saved: ~18,000 images x S$5.00 = S$90,000")
print(f"  Transfer learning: FREE (pre-trained weights are open-source)")



### Checkpoint 5


In [ ]:
assert (
    best_medical_transfer > 0.15
), f"Transfer with 2K data should beat random (acc={best_medical_transfer:.3f})"
# INTERPRETATION: With only 2,000 images, transfer learning dramatically
# outperforms training from scratch. In a medical context, this means
# fewer misdiagnosed patients and S$90,000+ saved in labelling costs.
# The pre-trained ResNet already knows edges, textures, and shapes —
# it only needs to learn which combinations indicate each skin condition.
print("\n--- Checkpoint 5 passed --- medical scenario complete\n")



## CLEANUP


In [ ]:
await conn.close()



## REFLECTION


[x] Loaded pre-trained ResNet-18 and froze the backbone
  [x] Trained only the classification head ({n_trainable:,} params vs {n_total:,} total)
  [x] Transfer vs scratch on full data: {best_transfer:.1%} vs {best_scratch:.1%}
  [x] Visualised layer activations: pre-trained (structured) vs random (noise)
  [x] Computed Grad-CAM heatmaps showing model attention
  [x] t-SNE cluster quality: transfer {cq_transfer:.3f} vs scratch {cq_scratch:.3f}
  [x] Medical scenario: {best_medical_transfer:.1%} transfer vs {best_medical_scratch:.1%} scratch with 2K images

  KEY INSIGHT: Transfer learning reuses visual features learned from
  1.28M ImageNet images. Like a doctor who studied general medicine
  before specialising, the model starts with deep knowledge instead of
  a blank slate. This is especially powerful when labelled data is
  scarce and expensive (medical images, industrial defects).

  NEXT: Part 3 quantifies data efficiency — how many labelled images
  do you actually need with transfer learning? This answers the VP's
  question: "How much do we need to spend on labelling?"


In [ ]:
print("\n" + "=" * 70)
print("  PART 2 COMPLETE — What You've Learned")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # Fine-tune last layer + optionally unfreeze
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (Transfer ResNet18 (ImageNet pretrained)) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        model,
        train_loader,
        _diag_loss,
        title="Transfer ResNet18 (ImageNet pretrained)",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): RMS 8.4e-04 to 3.2e-02 across frozen+unfrozen layers.
#     Frozen layers properly report 0 (no gradient by design).
# [✓] Dead neurons  (HEALTHY): 14% inactive — ImageNet-pretrained
#     weights + ReLU is the happy path.
# [✓] Loss trend    (HEALTHY): rapid convergence to 87% val accuracy in 5 epochs.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [BLOOD TEST] The HEALTHY RMS at every layer is the point of
#     transfer learning. ImageNet features are "warm-started" — no
#     vanishing gradients, no dead neurons, fast convergence.
#     Compare to ex_7/01 (43% dead conv1 from scratch).
#
#  [X-RAY — GRAD-CAM] Run diag.grad_cam(input, target_class,
#     layer_name='layer4') to see what the model attends to.
#     Before fine-tuning: attends to generic edges (ImageNet).
#     After fine-tuning: attends to Fashion-MNIST-specific features.
#     Slide 5G-ii Zech-2018-watermark example — ALWAYS check
#     attribution before deploying.
#
#  [STETHOSCOPE] 87% in 5 epochs vs 60% from scratch in 10 epochs
#     = 10x faster convergence, 30% better final accuracy.
#     Transfer learning is cheat code for small datasets.

